# Ensemble Analysis: Best Member Selection

## Objective
**Primary Goal:** Calculate distance (RMSE) from ground truth for each ensemble member and select the **10 best performing members**.

## Methodology
1. Load ensemble members from forecast directories
2. Load ground truth (real state at time=2)
3. **Compute RMSE for all individual members**
4. **Select best 10 members based on lowest RMSE**
5. Save best 10 members and their forecasts

## Additional Analysis
- RMSE per channel (investigative): Validate that ocean physics are within expected ranges
- Ensemble statistics: Mean and variance (for reference)
- Visualizations: Compare best members to ground truth

---

In [ ]:
import xarray  as xr

In [ ]:
import os
import glob
import numpy as np
from pathlib import Path

# Define forecast directories
forecast_dirs = {
    # 'onevector': './onevector_forecast',
    # 'pointwise1': './pointwise1_forecast',
    # 'pointwise2': './pointwise2_forecast'
    'pointwise_1' : './outputs/ens_init2/pointwise_1'
}

# Check which directories exist
existing_dirs = {}
for name, path in forecast_dirs.items():
    if os.path.exists(path):
        existing_dirs[name] = path
        print(f"✓ Found {name}: {path}")
    else:
        print(f"✗ Not found: {path}")

print(f"\n{len(existing_dirs)} out of {len(forecast_dirs)} directories found")

## Step 1: Load Ensemble Members from All Directories

In [ ]:
def load_ensemble_members(directory):
    """
    Load all NetCDF files from a directory as ensemble members.
    Returns a list of xarray datasets.
    """
    if not os.path.exists(directory):
        print(f"Warning: Directory {directory} does not exist")
        return []
    
    # Find all .nc files
    nc_files = sorted(glob.glob(os.path.join(directory, "*.nc")))
    
    if not nc_files:
        print(f"Warning: No NetCDF files found in {directory}")
        return []
    
    print(f"Loading {len(nc_files)} files from {directory}...")
    members = []
    for nc_file in nc_files:
        try:
            ds = xr.open_dataset(nc_file)
            members.append(ds)
            print(f"  ✓ Loaded: {os.path.basename(nc_file)}")
        except Exception as e:
            print(f"  ✗ Error loading {nc_file}: {e}")
    
    return members

# Load all ensemble members
ensemble_data = {}
for name, path in existing_dirs.items():
    print(f"\n--- Loading {name} ensemble ---")
    ensemble_data[name] = load_ensemble_members(path)
    print(f"Total members loaded: {len(ensemble_data[name])}")

## Step 2: Load Real State (Ground Truth)

In [ ]:
# Load the real state at time=2
real_state_path = "/Odyssey/public/glonet/glorys12_1993-01-01_to_1993-06-30_init_states"

print(f"Loading real state from: {real_state_path}")
try:
    # Load the combined input file
    real_state_ds = xr.open_dataset(
        os.path.join(real_state_path, "combined_input.nc"),
        chunks=96
    )
    
    # Extract time=2 (ground truth for forecasts starting at time=0)
    real_state = real_state_ds.isel(time=2).persist()
    
    print(f"✓ Real state loaded successfully")
    print(f"  Shape: {real_state['data'].shape}")
    print(f"  Variables: {list(real_state.data_vars)}")
    print(f"  Coordinates: {list(real_state.coords)}")
    
except Exception as e:
    print(f"✗ Error loading real state: {e}")
    real_state = None

## Step 3: Extract time=0 from All Ensemble Members

In [ ]:
# Extract time=0 from each ensemble member
ensemble_t0 = {}

for name, members in ensemble_data.items():
    print(f"\n--- Processing {name} ensemble ---")
    
    if not members:
        print(f"  No members to process")
        continue
    
    t0_list = []
    for i, member in enumerate(members):
        try:
            # Extract time=0 (assuming 'time' dimension exists)
            if 'time' in member.dims:
                t0 = member.isel(time=0)
            else:
                t0 = member
            
            t0_list.append(t0)
            print(f"  ✓ Member {i+1}: extracted time=0")
        except Exception as e:
            print(f"  ✗ Member {i+1}: error - {e}")
    
    ensemble_t0[name] = t0_list
    print(f"Total time=0 slices: {len(t0_list)}")

print(f"\n{'='*60}")
print(f"Summary: {sum(len(v) for v in ensemble_t0.values())} total time=0 slices across all ensembles")

## Step 4: Compute Ensemble Statistics (Mean and Variance)

In [ ]:
def compute_ensemble_statistics(members_list, name):
    """
    Compute mean and variance across ensemble members.
    
    Args:
        members_list: List of xarray datasets (ensemble members at time=0)
        name: Name of the ensemble for printing
    
    Returns:
        dict with 'mean' and 'variance' datasets
    """
    if not members_list:
        print(f"No members for {name}")
        return None
    
    print(f"\n--- Computing statistics for {name} ensemble ---")
    print(f"Number of members: {len(members_list)}")
    
    # Stack all members along a new 'member' dimension
    # Assuming all members have a 'data' variable
    try:
        # Extract the data arrays and stack them
        data_arrays = []
        # Get real_state data
        real_data = real_state['data'] if isinstance(real_state, xr.Dataset) and 'data' in real_state else real_state
        
        for i, member in enumerate(members_list):
            if 'data' in member:
                data_arrays.append((member['data'] - real_data).fillna(0))
            else:
                print(f"Warning: Member {i} doesn't have 'data' variable")
                print(f"Available variables: {list(member.data_vars)}")
        
        if not data_arrays:
            print("No valid data arrays found")
            return None
        
        # Concatenate along new 'member' dimension
        stacked = xr.concat(data_arrays, dim='member')
        
        # Compute overall statistics
        ensemble_mean = stacked.mean(dim='member', skipna=True)
        ensemble_var = stacked.var(dim='member', skipna=True)
        ensemble_std = stacked.std(dim='member', skipna=True)
        
        print(f"✓ Statistics computed")
        print(f"  Shape: {ensemble_mean.sizes}")
        print(f"  Dimensions: {list(ensemble_mean.dims)}")
        
        # Determine channel dimension name (could be 'ch' or 'channel')
        channel_dim = None
        for dim in ['ch', 'channel']:
            if dim in ensemble_mean.dims:
                channel_dim = dim
                break
        
        # Compute per-channel statistics
        if channel_dim is not None:
            n_channels = ensemble_mean.sizes[channel_dim]
            print(f"\n  Per-Channel Statistics ({n_channels} channels):")
            print(f"  {'Channel':<10} {'Mean':<15} {'Std Dev':<15} {'Min':<15} {'Max':<15}")
            print(f"  {'-'*70}")
            
            for ch in range(n_channels):
                ch_mean = ensemble_mean.isel({channel_dim: ch})
                ch_std = ensemble_std.isel({channel_dim: ch})
                
                # Compute statistics across spatial dimensions (skipna for land mask)
                mean_val = float(ch_mean.mean(skipna=True))
                std_val = float(ch_std.mean(skipna=True))
                min_val = float(ch_mean.min(skipna=True))
                max_val = float(ch_mean.max(skipna=True))
                
                print(f"  {ch:<10} {mean_val:<15.6f} {std_val:<15.6f} {min_val:<15.6f} {max_val:<15.6f}")
                
            # Overall summary
            print(f"\n  Overall Statistics:")
            print(f"    Mean value range: [{float(ensemble_mean.min()):.4f}, {float(ensemble_mean.max()):.4f}]")
            print(f"    Variance value range: [{float(ensemble_var.min()):.6f}, {float(ensemble_var.max()):.6f}]")
            print(f"    Std dev value range: [{float(ensemble_std.min()):.6f}, {float(ensemble_std.max()):.6f}]")
        else:
            # If no channel dimension, show overall statistics
            print(f"  Mean value range: [{float(ensemble_mean.min()):.4f}, {float(ensemble_mean.max()):.4f}]")
            print(f"  Variance value range: [{float(ensemble_var.min()):.6f}, {float(ensemble_var.max()):.6f}]")
            print(f"  Std dev value range: [{float(ensemble_std.min()):.6f}, {float(ensemble_std.max()):.6f}]")
        
        return {
            'mean': ensemble_mean,
            'variance': ensemble_var,
            'std': ensemble_std,
            'members': stacked
        }
    
    except Exception as e:
        print(f"✗ Error computing statistics: {e}")
        import traceback
        traceback.print_exc()
        return None

# Compute statistics for each ensemble
ensemble_stats = {}
for name, members in ensemble_t0.items():
    stats = compute_ensemble_statistics(members, name)
    if stats is not None:
        ensemble_stats[name] = stats

## Visualize Standard Deviation for 5 Surface Variables

Visualize the ensemble standard deviation for:
- **SSH** (Sea Surface Height) - Channel 0
- **SST** (Sea Surface Temperature) - Channel 1  
- **SSS** (Sea Surface Salinity) - Channel 2
- **uo** (Surface zonal velocity) - Channel 3
- **vo** (Surface meridional velocity) - Channel 4

In [ ]:
import matplotlib.pyplot as plt

# Define the 5 surface variables
surface_vars = [
    ('SSH', 0, 'Sea Surface Height'),
    ('SST', 1, 'Sea Surface Temperature'),
    ('SSS', 2, 'Sea Surface Salinity'),
    ('uo', 3, 'Surface Zonal Velocity'),
    ('vo', 4, 'Surface Meridional Velocity')
]

# Create visualization for each ensemble
for ensemble_name, stats in ensemble_stats.items():
    print(f"\n{'='*80}")
    print(f"Visualizing standard deviation for {ensemble_name} ensemble")
    print(f"{'='*80}\n")
    
    std = stats['std']
    
    # Determine channel dimension name
    channel_dim = None
    for dim in ['ch', 'channel']:
        if dim in std.dims:
            channel_dim = dim
            break
    
    if channel_dim is None:
        print(f"Warning: No channel dimension found for {ensemble_name}")
        continue
    
    # Check if we have lat/lon coordinates
    has_spatial = 'lat' in std.dims and 'lon' in std.dims
    
    # Create figure with subplots
    fig, axes = plt.subplots(3, 2, figsize=(16, 14))
    axes = axes.flatten()
    
    for idx, (var_name, ch_idx, var_desc) in enumerate(surface_vars):
        ax = axes[idx]
        
        # Extract standard deviation for this channel
        var_std = std.isel({channel_dim: ch_idx})
        
        # Compute statistics
        var_min = float(var_std.min(skipna=True))
        var_max = float(var_std.max(skipna=True))
        var_mean = float(var_std.mean(skipna=True))
        var_std_val = float(var_std.std(skipna=True))
        
        print(f"{var_name} ({var_desc}):")
        print(f"  Min std dev:  {var_min:.6e}")
        print(f"  Max std dev:  {var_max:.6e}")
        print(f"  Mean std dev: {var_mean:.6e}")
        print(f"  Std dev of std dev:  {var_std_val:.6e}\n")
        
        if has_spatial:
            # Spatial plot
            # Get lat/lon coordinates
            lats = var_std['lat'].values
            lons = var_std['lon'].values
            
            # Plot the standard deviation
            im = ax.pcolormesh(lons, lats, var_std.values,
                              cmap='viridis', shading='auto')
            
            ax.set_xlabel('Longitude', fontsize=10)
            ax.set_ylabel('Latitude', fontsize=10)
            ax.set_aspect('equal', adjustable='box')
            
            # Colorbar
            cbar = plt.colorbar(im, ax=ax, orientation='vertical', pad=0.02, shrink=0.8)
            cbar.set_label(f'Std Dev', fontsize=9)
            
            # Title with statistics
            ax.set_title(f'{var_name}: {var_desc}\nMean: {var_mean:.3e}, Range: [{var_min:.3e}, {var_max:.3e}]', 
                        fontsize=11, fontweight='bold')
        else:
            # Fallback: histogram if no spatial dimensions
            var_data_flat = var_std.values.flatten()
            var_data_flat = var_data_flat[~np.isnan(var_data_flat)]
            
            ax.hist(var_data_flat, bins=50, alpha=0.7, color='blue', edgecolor='black')
            ax.axvline(var_mean, color='red', linestyle='--', linewidth=2, label=f'Mean: {var_mean:.3e}')
            ax.set_xlabel('Std Dev', fontsize=10)
            ax.set_ylabel('Frequency', fontsize=10)
            ax.set_title(f'{var_name}: {var_desc}', fontsize=11, fontweight='bold')
            ax.legend()
            ax.grid(True, alpha=0.3)
    
    # Hide the 6th subplot (we only have 5 variables)
    axes[5].axis('off')
    
    # Overall title
    fig.suptitle(f'Ensemble Standard Deviation: {ensemble_name.upper()}', 
                 fontsize=16, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n{'='*80}\n")

## Step 5: Compute Distance to Real State

In [ ]:
def compute_distance_metrics(forecast, real_state, name):
    """
    Compute various distance metrics between forecast and real state.
    
    Metrics:
    - MSE: Mean Squared Error
    - RMSE: Root Mean Squared Error
    - MAE: Mean Absolute Error
    - L2 norm: Euclidean distance
    
    Args:
        forecast: xarray DataArray (forecast at time=0)
        real_state: xarray DataArray (real state at time=2)
        name: Name for printing
    
    Returns:
        dict with distance metrics
    """
    print(f"\n--- Computing distances for {name} ---")
    
    try:
        # Get the 'data' variable if it exists
        if isinstance(forecast, xr.Dataset):
            forecast_data = forecast['data'] if 'data' in forecast else forecast
        else:
            forecast_data = forecast
            
        if isinstance(real_state, xr.Dataset):
            real_data = real_state['data'] if 'data' in real_state else real_state
        else:
            real_data = real_state
        
        # Compute difference
        diff = forecast_data - real_data
        
        # Compute metrics (skipna=True to handle land mask)
        mse = float((diff ** 2).mean(skipna=True))
        rmse = np.sqrt(mse)
        mae = float(np.abs(diff).mean(skipna=True))
        l2_norm = float(np.sqrt((diff ** 2).sum(skipna=True)))
        
        # Count valid (ocean) pixels
        n_valid = int((~np.isnan(diff)).sum())
        n_total = int(diff.size)
        
        # Spatial metrics (per channel if applicable)
        spatial_mse = (diff ** 2).mean(dim=['lat', 'lon'], skipna=True) if 'lat' in diff.dims and 'lon' in diff.dims else None
        
        print(f"✓ Distance metrics computed (land mask applied)")
        print(f"  Valid pixels: {n_valid}/{n_total} ({100*n_valid/n_total:.1f}% ocean)")
        print(f"  MSE:  {mse:.6e}")
        print(f"  RMSE: {rmse:.6e}")
        print(f"  MAE:  {mae:.6e}")
        print(f"  L2 norm: {l2_norm:.6e}")
        
        return {
            'mse': mse,
            'rmse': rmse,
            'mae': mae,
            'l2_norm': l2_norm,
            'spatial_mse': spatial_mse,
            'difference': diff
        }
    
    except Exception as e:
        print(f"✗ Error computing distances: {e}")
        return None

# Compute distances for ensemble means (for comparison purposes)
if real_state is not None:
    distance_metrics = {}
    
    print("\n" + "="*70)
    print("REFERENCE: Ensemble Mean Distances")
    print("="*70)
    
    for name, stats in ensemble_stats.items():
        metrics = compute_distance_metrics(stats['mean'], real_state, f"{name} (ensemble mean)")
        if metrics is not None:
            distance_metrics[f"{name}_mean"] = metrics
else:
    print("Real state not loaded, skipping distance computation")
    distance_metrics = {}

## Step 6: Compute Individual Member Distances and Select Best 10

**Primary Goal:** For each ensemble member, compute RMSE to ground truth and identify the 10 best performers.

In [ ]:
def compute_member_mse(member, real_state):
    """
    Compute MSE between a single member and real state.
    
    Args:
        member: xarray Dataset (single ensemble member at time=0)
        real_state: xarray Dataset (ground truth)
    
    Returns:
        float: MSE value
    """
    # Get the 'data' variable
    if isinstance(member, xr.Dataset):
        member_data = member['data'] if 'data' in member else member
    else:
        member_data = member
        
    if isinstance(real_state, xr.Dataset):
        real_data = real_state['data'] if 'data' in real_state else real_state
    else:
        real_data = real_state
    
    # Compute MSE (skipna=True to handle land mask)
    diff = member_data - real_data
    mse = float((diff ** 2).mean(skipna=True))
    
    return mse


# Compute MSE for all individual members
print("="*80)
print("COMPUTING MSE FOR ALL INDIVIDUAL MEMBERS")
print("="*80)

all_member_results = []

if real_state is not None:
    for ensemble_name, members in ensemble_t0.items():
        print(f"\n--- Processing {ensemble_name} ensemble ({len(members)} members) ---")
        
        for idx, member in enumerate(members):
            try:
                mse = compute_member_mse(member, real_state)
                rmse = np.sqrt(mse)
                
                # Store results
                result = {
                    'ensemble': ensemble_name,
                    'member_idx': idx,
                    'mse': mse,
                    'rmse': rmse,
                    'member_data': member  # Keep reference to the member
                }
                all_member_results.append(result)
                
                if (idx + 1) % 10 == 0:
                    print(f"  Processed {idx + 1}/{len(members)} members...")
                
            except Exception as e:
                print(f"  ✗ Error processing member {idx}: {e}")
        
        print(f"  ✓ Finished {ensemble_name}: {len(members)} members processed")
    
    print(f"\n{'='*80}")
    print(f"Total members processed: {len(all_member_results)}")
    
    # Sort by MSE (ascending - best performers first)
    all_member_results.sort(key=lambda x: x['mse'])
    
    # Select top 10 best members
    best_10_members = all_member_results[:10]
    
    print(f"\n{'='*80}")
    print("TOP 10 BEST PERFORMING MEMBERS (Lowest MSE)")
    print("="*80)
    print(f"{'Rank':<6} {'Ensemble':<15} {'Member':<10} {'MSE':<18} {'RMSE':<18}")
    print("-"*80)
    
    for rank, result in enumerate(best_10_members, 1):
        print(f"{rank:<6} {result['ensemble']:<15} {result['member_idx']:<10} "
              f"{result['mse']:<18.10e} {result['rmse']:<18.10e}")
    
    print(f"\n{'='*80}")
    print(f"Best MSE:  {best_10_members[0]['mse']:.10e}")
    print(f"Worst (of top 10) MSE: {best_10_members[-1]['mse']:.10e}")
    print(f"Improvement ratio: {best_10_members[-1]['mse'] / best_10_members[0]['mse']:.4f}x")
    
    # Compare with ensemble mean
    if distance_metrics:
        ensemble_mean_mse = distance_metrics.get('total_mean', {}).get('mse', None)
        if ensemble_mean_mse:
            print(f"\nEnsemble mean MSE: {ensemble_mean_mse:.10e}")
            print(f"Best member is {ensemble_mean_mse / best_10_members[0]['mse']:.4f}x better than ensemble mean")
    
else:
    print("Real state not loaded, cannot compute member distances")
    best_10_members = []

## Step 7: Visualize MSE Distribution and Analyze Best Members

In [ ]:
import matplotlib.pyplot as plt

# Extract MSE values for analysis
all_mse_values = [r['mse'] for r in all_member_results]
best_10_mse = [r['mse'] for r in best_10_members]

print("="*80)
print("MSE DISTRIBUTION STATISTICS")
print("="*80)
print(f"Total members: {len(all_mse_values)}")
print(f"Mean MSE:      {np.mean(all_mse_values):.10e}")
print(f"Median MSE:    {np.median(all_mse_values):.10e}")
print(f"Std Dev MSE:   {np.std(all_mse_values):.10e}")
print(f"Min MSE:       {np.min(all_mse_values):.10e} (best)")
print(f"Max MSE:       {np.max(all_mse_values):.10e} (worst)")
print(f"\nTop 10 MSE range: [{best_10_mse[0]:.10e}, {best_10_mse[-1]:.10e}]")
print(f"Top 10 mean:      {np.mean(best_10_mse):.10e}")

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of all MSE values
axes[0].hist(all_mse_values, bins=30, alpha=0.7, color='blue', edgecolor='black')
axes[0].axvline(best_10_mse[-1], color='red', linestyle='--', linewidth=2, label='Top 10 threshold')
axes[0].axvline(np.mean(all_mse_values), color='green', linestyle='--', linewidth=2, label='Mean MSE')
axes[0].set_xlabel('MSE', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('MSE Distribution Across All Members', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Sorted MSE values with top 10 highlighted
sorted_indices = np.arange(len(all_mse_values))
axes[1].plot(sorted_indices, all_mse_values, 'o-', alpha=0.6, markersize=4, label='All members')
axes[1].plot(sorted_indices[:10], best_10_mse, 'ro', markersize=8, label='Top 10 best')
axes[1].set_xlabel('Rank (sorted by MSE)', fontsize=12)
axes[1].set_ylabel('MSE', fontsize=12)
axes[1].set_title('Ranked Members by MSE Performance', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("BEST 10 MEMBERS SUMMARY")
print("="*80)
print("\nMember IDs of top 10 performers:")
best_member_ids = [r['member_idx'] for r in best_10_members]
print(f"  {best_member_ids}")
print(f"\nThese members will be used for ensemble-based optimization.")

# Create a dictionary for easy access to best members
best_members_dict = {
    f"member_{r['member_idx']:03d}": {
        'rank': rank,
        'ensemble': r['ensemble'],
        'member_idx': r['member_idx'],
        'mse': r['mse'],
        'rmse': r['rmse'],
        'data': r['member_data']
    }
    for rank, r in enumerate(best_10_members, 1)
}

print(f"\nBest members stored in 'best_members_dict' for further analysis.")

## Step 8: Save Best 10 Members

In [ ]:
# Create output directory for best members
output_dir = forecast_dirs['pointwise_1'] + "/best_10_members"
os.makedirs(output_dir, exist_ok=True)

print("="*80)
print("SAVING BEST 10 MEMBERS")
print("="*80)
print(f"Output directory: {output_dir}\n")

# Save the full forecast files for each of the best 10 members
for rank, result in enumerate(best_10_members, 1):
    ensemble_name = result['ensemble']
    member_idx = result['member_idx']
    
    # Get the original file path
    original_file = os.path.join(existing_dirs[ensemble_name], 
                                  f"member_{member_idx:03d}_initial_condition.nc")
    
    # Create output filename
    output_file = os.path.join(output_dir, 
                                f"rank{rank:02d}_member_{member_idx:03d}_mse_{result['mse']:.6e}.nc")
    
    try:
        # Load the full forecast (all timesteps, not just time=0)
        full_forecast = xr.open_dataset(original_file)
        
        # Save to new location
        full_forecast.to_netcdf(output_file)
        
        print(f"✓ Rank {rank}: member_{member_idx:03d} → {os.path.basename(output_file)}")
        print(f"   MSE: {result['mse']:.10e}, RMSE: {result['rmse']:.10e}")
        
    except Exception as e:
        print(f"✗ Error saving member {member_idx}: {e}")

# Save metadata summary
metadata_file = os.path.join(output_dir, 'best_members_summary.txt')
with open(metadata_file, 'w') as f:
    f.write("="*80 + "\n")
    f.write("TOP 10 BEST PERFORMING ENSEMBLE MEMBERS\n")
    f.write("="*80 + "\n\n")
    f.write(f"Selection criterion: Lowest MSE to ground truth\n")
    f.write(f"Ground truth: time=2 from combined_input.nc\n\n")
    f.write(f"{'Rank':<6} {'Member':<10} {'MSE':<20} {'RMSE':<20}\n")
    f.write("-"*80 + "\n")
    
    for rank, result in enumerate(best_10_members, 1):
        f.write(f"{rank:<6} {result['member_idx']:<10} {result['mse']:<20.10e} {result['rmse']:<20.10e}\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write(f"Best MSE:  {best_10_members[0]['mse']:.10e}\n")
    f.write(f"Worst (of top 10) MSE: {best_10_members[-1]['mse']:.10e}\n")
    f.write(f"Mean MSE (all 80): {np.mean(all_mse_values):.10e}\n")

print(f"\n✓ Metadata saved to: {metadata_file}")
print("\n" + "="*80)
print("SAVE COMPLETE")
print("="*80)
print(f"\nBest 10 members saved to: {output_dir}")
print(f"Total files: {len(best_10_members)} NetCDF files + 1 summary")